# Tune camera focus

Your camera might not focus well. Turn the ring to adjust the focus

https://www.youtube.com/watch?v=njbqPSWnuWw

In [1]:
import cv2
import time
import ipywidgets as widgets
from IPython.display import display
from picamera2 import Picamera2
import logging
import threading

# 設定 picamera2 的日誌等級，只顯示錯誤
Picamera2.set_logging(logging.ERROR)

def stream_video_minimal(stop_button, image_widget):
    # 'try...finally' 確保相機被正確釋放
    try:
        # 'with' 陳述式會自動開啟和關閉相機
        with Picamera2() as picam2:
            # 建立一個 640x480 的相機設定
            config = picam2.create_preview_configuration(main={"size": (640, 480)})
            # 套用設定
            picam2.configure(config)
            # 啟動相機
            picam2.start()
            
            # 當停止按鈕的 .value 不是 True (即未被按下) 時，持續迴圈
            while not stop_button.value:
                # 從相機擷取一個畫面 (RGBA 格式)
                frame = picam2.capture_array()
                # 將顏色從 RGBA 轉換為 BGR (OpenCV 預設格式)
                frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGBA2BGR)
                # 翻轉影像 (參數 -1 表示水平和垂直翻轉)
                frame_flipped = cv2.flip(frame_bgr, -1)
                
                # 將影像幀編碼為 JPEG 格式
                _, buffer = cv2.imencode('.jpeg', frame_flipped)
                # 更新 image widget 的內容 (必須傳入位元組)
                image_widget.value = buffer.tobytes()
                
                # 短暫暫停，釋放 CPU 資源
                time.sleep(0.01)

    finally:
        # 當迴圈停止 (按鈕被按下) 時，清除影像
        image_widget.value = b''

# --- 主程式 ---

# 建立一個 ToggleButton (開/關按鈕)
stopButton = widgets.ToggleButton(
    value=False, description='Stop', button_style='danger', icon='square'
)
# 建立一個用於顯示 JPEG 影像的 Image widget
image_widget = widgets.Image(format='jpeg', width=640, height=480)

# 使用 VBox 垂直佈局來顯示 widget
display(widgets.VBox([image_widget, stopButton]))

# 建立一個背景執行緒，執行 stream_video_minimal 函式
thread = threading.Thread(
    target=stream_video_minimal, 
    args=(stopButton, image_widget) # 傳遞 widget 物件給函式
)
# 啟動執行緒
thread.start()

[22:59:15.958294786] [4698]  INFO Camera camera_manager.cpp:330 libcamera v0.5.2+99-bfd68f78
[22:59:15.965575951] [4699]  INFO RPI pisp.cpp:720 libpisp version v1.2.1 981977ff21f3 29-04-2025 (14:13:50)
[22:59:15.968148364] [4699]  INFO IPAProxy ipa_proxy.cpp:180 Using tuning file /usr/share/libcamera/ipa/rpi/pisp/ov5647.json
[22:59:15.975002121] [4699]  INFO Camera camera_manager.cpp:220 Adding camera '/base/axi/pcie@1000120000/rp1/i2c@88000/ov5647@36' for pipeline handler rpi/pisp
[22:59:15.975017899] [4699]  INFO RPI pisp.cpp:1179 Registered camera /base/axi/pcie@1000120000/rp1/i2c@88000/ov5647@36 to CFE device /dev/media0 and ISP device /dev/media2 using PiSP variant BCM2712_C0
[22:59:15.979093131] [4698]  INFO Camera camera.cpp:1215 configuring streams: (0) 640x480-XBGR8888/sRGB (1) 640x480-GBRG_PISP_COMP1/RAW
[22:59:15.979190668] [4699]  INFO RPI pisp.cpp:1483 Sensor: /base/axi/pcie@1000120000/rp1/i2c@88000/ov5647@36 - Selected sensor format: 640x480-SGBRG10_1X10/RAW - Selected CF